In [0]:
%sh apt-get install -y osmium-tool

In [0]:
import subprocess
import json
import os

# 1. 경로 설정
input_pbf = "/dbfs/mnt/bronze/osm/raw/south-korea-latest.osm.pbf"
output_dir = "/dbfs/mnt/silver/seoul_districts"
config_file = "extract_config.json"

os.makedirs(output_dir, exist_ok=True)

# 2. Osmium Multi-extract용 설정 파일 생성
# 한 번의 실행으로 3개 구를 동시에 추출하도록 정의합니다.
extracts_config = {
    "extracts": [
        {
            "output": "gangnam.pbf",
            "bbox": [127.01, 37.45, 127.12, 37.54],
            "description": "Gangnam-gu extraction"
        },
        {
            "output": "seocho.pbf",
            "bbox": [126.97, 37.43, 127.09, 37.52],
            "description": "Seocho-gu extraction"
        },
        {
            "output": "songpa.pbf",
            "bbox": [127.07, 37.46, 127.18, 37.54],
            "description": "Songpa-gu extraction"
        }
    ],
    "directory": output_dir
}

with open(config_file, 'w') as f:
    json.dump(extracts_config, f)

# 3. Multi-extract 명령어 실행
print("🚀 Starting Multi-extract process...")
cmd = [
    "osmium", "extract",
    "--config", config_file,
    "--strategy", "complete_ways",
    input_pbf,
    "--overwrite"
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    print("✅ Success! All 3 districts extracted in a single pass.")
    print(f"Files are located in: {output_dir}")
else:
    print(f"❌ Error: {result.stderr}")

In [0]:
# NOTE:
# Data is stored in Azure Blob Storage and mounted at /mnt/raw in Databricks.
# Mounting code is omitted for security reasons.

# Example path usage:
raw_path = "/mnt/raw/seoul_api"

# (Optional) list files if environment supports it
# dbutils.fs.ls(raw_path)

In [0]:
df_raw = spark.read.option("multiline", "true").json("/mnt/raw/seoul_api/citydata_20260408_033105.json")
display(df_raw)

In [0]:
dbutils.fs.mount(
    source = "wasbs://silver@3dtstorage.blob.core.windows.net/",
    mount_point = "/mnt/silver",
    extra_configs = {"fs.azure.account.key.3dtstorage.blob.core.windows.net": storage_key}
)
print("silver 마운트 완료!")